# Metasyn multiple table tutorial

In this tutorial you will learn how to create synthetic versions of multiple tables at onc, while preserving some relations between tables.

First you should install metasyn if you have not done so already.

In [ ]:
# %pip install metasyn

### Loading the dataset

We will use a demonstration dataset that is built into metasyn, called `ShopMultiDataset`. This dataset contains three tables that are interrelated through customer id's and product id's.



In [ ]:
from metasyn.demo.dataset import ShopMultiDataset
from metasyn.multiframe import MultiFrame

Here we load in our demo dataset. In your own usecase, you simply need to read the files into
a dictionary of data frames.
For example: ``data = {"name_1": pl.read_csv("file1.csv"), ...}``

In [ ]:
data = ShopMultiDataset().get_dataframes()
print({key: d.head(1) for key, d in data.items()})

The dataset of tables is constructed so that we can join the different tables. Let's try that out now.

In [ ]:
data["purchases"].join(data["customers"], left_on="customer_id", right_on="id", validate="m:1")

### Synthesizing unrelated tables

Now, let us naively generate synthetic data independently using metasyn without specifying any relations.

In [ ]:
multiframe = MultiFrame.fit_dataframes(data, relations=[])
syn_data = multiframe.synthesize()
# Try to join the same tables
syn_data["purchases"].join(syn_data["customers"], left_on="customer_id", right_on="id", validate="m:1")

Note above that while the synthetic data has the same number of rows for the tables, the number of rows in the joined table is vastly different (any results are mostly due to luck or lack thereof).

### Synthesizing related tables
To remedy this, we can specify relations in the dataset.

In [ ]:
relations = [
    "customers[id] <? purchases[customer_id]",
    "products[id] <? purchases[product_id]",
]

multiframe_improved = MultiFrame.fit_dataframes(data, relations=relations)
syn_data_improved = multiframe_improved.synthesize()
syn_data_improved["purchases"].join(syn_data_improved["customers"], left_on="customer_id", right_on="id", validate="m:1")

We can also adjust the size of the output tables for each individual table:

In [ ]:
multiframe_improved.synthesize(n={"customers": 8})

### Saving and loading multiframes

Similar to metaframes, multiframes can also be saved and loaded from a .json file.

In [ ]:
multiframe.save_json("test.json")
mf = multiframe.load_json("test.json")